# N8N
La clase de hoy no tiene que ver directamente con Python pero busca familiarisarlos con el uso de nuevas herramientas `low-code` o `no-code`. Estas nuevas herramientas permiten construir automatizaciones, agentes y aplicaciones sin necesidad de saber programar, pero tener unas bases solidas en programación en conjunto del uso de estas nuevas tecnologías puede ser un multiplicador de productividad para sus proyectos. 

El día de hoy conoceremos **n8n** (de "node for node"), una startup fundada en Berlín en 2019 que ofrece una herramienta visual para automatizar tareas y conectar servicios como Gmail, Google Sheets, Slack, APIs o bases de datos, sin necesidad de escribir backend ni levantar servidores. Aunque puede usarse sin saber programar, quienes tienen conocimientos técnicos pueden personalizar los flujos con lógica condicional, expresiones, y bloques en JavaScript o Python.

Se puede utilizar `n8n` para:
- Crear un flujo que reciba respuestas de un formulario, las analice con inteligencia artificial, las guarde en una hoja de cálculo y envíe una notificación automática al equipo.
- Crear un chatbot para Whatsapp o Telegram que agende las citas de un consultorio medico y utilice IA para clonar la voz de el personal medico para responder mediante notas de voz.
- Crear un asistente personal que te notifique las tareas del día y te organice un cronograma para el día teniendo en cuenta tu calendario.

Pero, ojo, entre sus limitaciones están:
- No reemplaza un backend complejo (por ejemplo, no está hecho para manejar miles de usuarios concurrentes o lógica empresarial crítica).
- Aunque puede usarse sin programar, sacar el máximo provecho sí requiere entender conceptos como APIs, webhooks o transformaciones de datos.
- Algunos flujos pueden volverse difíciles de mantener si se abusa de la lógica condicional sin modularidad.

Antes de comenzar, vamos a introducir el concepto de API que va a ayudar a entender mejor cómo funciona n8n y otras aplicaciones que conectan diferentes servicios (Gmail, ChatGPT, Whatsapp)

# Qué es una API?
Una **API** (Application Programming Interface) es una forma en la que dos programas pueden **comunicarse entre sí**. Es como un **menú de opciones** que un servicio ofrece para que tú puedas pedirle información o enviarle datos.

Una buena analogía podría ser un restaurante donde el menú es la API. Tú eliges un plato (por ejemplo, "quiero los datos del clima") y el camarero (la API) lleva tu pedido a la cocina (el servidor) y luego te trae la comida (los datos).

Las APIs permiten:

- Obtener datos en tiempo real desde servicios como Twitter, bancos, clima, etc.
- Enviar información para procesarla (por ejemplo, enviar texto a un modelo de IA).
- Automatizar tareas que normalmente harías manualmente.

## ¿Cómo se usan en Python?
Generalmente, se utiliza la librería `requests` en Python para hacer peticiones a una API. A continuación, veremos un ejemplo sencillo usando [open-meteo](https://open-meteo.com), una API meteorológica gratuita que permite consultar datos del clima. Esta API ofrece información actual, histórica y pronósticos de alta resolución, con datos disponibles en grillas espaciales que van de 1 a 11 kilómetros.

Algunos conceptos claves sobre los `requests` son:
- `GET`: pedir datos.
- `POST`: enviar datos.
- `URL`: la dirección a donde haces la petición.
- `Headers`: información extra (como tu token de acceso).
- `JSON`: formato común para intercambiar datos entre servicios.

In [ ]:
import requests

lat_bogota = 4.661689
lon_bogota = -74.085017

response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat_bogota}&longitude={lon_bogota}&current_weather=true")

data = response.json()
# Noten que generalmente el output de un request a una API es un json (o sea un diccionario en Python)
data

{'latitude': 4.625,
 'longitude': -74.125,
 'generationtime_ms': 0.03719329833984375,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 2553.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2025-05-03T16:15',
  'interval': 900,
  'temperature': 16.3,
  'windspeed': 13.2,
  'winddirection': 125,
  'is_day': 1,
  'weathercode': 3}}

Ahora que sabemos que son las APIs y entendemos el concepto, podemos sacarle el máximo provecho a `n8n`. 

El día de hoy haremos una integración sencilla. **Vamos a conectar WhatsApp con ChatGPT** de forma que tengamos acceso a un chat con nuestro asistente personal. 

# Tutorial para conectar WhatsApp con ChatGPT usando n8n

## Paso 1. Crear una cuenta en `n8n`.
1. Acceder al landing page de [n8n](https://n8n.io) y darle click al boton en la esquina superior derecha que dice **Get Started**.
2. Nos saldrá un formulario donde nos solicita unos datos de registro con los cuales iniciaremos una prueba gratuita de 14 días. 
3. Continuar completando las preguntas hasta llegar al panel general o **Overview** de n8n.

<center>
<img src='./img/overview.png' width=60%>
</center>

## Paso 2. Crear un nuevo proyecto desde cero.
1. Una vez estemos en la ventana de Overview, presionaremos el icono que dice **Start from scratch**.
<center>
<img src='./img/start_from_scratch.png' width=60%>
</center>

2. Ahora añadiremos el paso a paso de nuestro flujo haciendo click en el botón que dice **Add first step...**. Crearemos todo el flujo primero y luego lo configuraremos.

    - Primero añadiremos un *trigger* que consista en recibir un mensaje de WhatsApp. Para ello, luego de darle click a **Add first step...**, veremos que se abre una venta a la derecha de la pantalla. En la barra de búsqueda que dice "Search nodes..." vamos a escribir "Whatsapp". Seleccionamos **WhatsApp Business Cloud** y luego "On messages".
    <center>
    <img src='./img/whatsapp1.png' width=30%>
    <img src='./img/whatsapp2.png' height=500px>
    </center>
    
    - Una vez se abra el menu de "WhatsApp Trigger", lo vamos a ignorar por ahora y vamos a presionar **Back to canvas** en la esquina superior izquierda. Nuestro *workflow* se debería ver algo así:
    <center>
    <img src='./img/whatsapp_trigger.png' width=20%>
    </center>
    
    - Continuaremos dando click en el pequeño botón con un **+** a la derecha de Whatsapp Trigger para añadir un agente IA. Seleccionamos **AI** y luego **AI Agent**. Ignoramos la ventana de configuración y presionamos **Back to canvas**.
    <center>
    <img src='./img/AI1.png' height=500px>
    <img src='./img/AI2.png' height=500px>
    </center>
        El resultado se debería ver como:
    <center>
    <img src='./img/AI_agent1.png' width=20%>
    </center>

    - Ahora debemos escoger un LLM y una memoria. Para el LLM presionamos el **+** debajo de "Chat Model". En la ventana que nos aparece escribimos "OpenAI" y seleccionamos la opción "OpenAI Chat Model". Ignoramos la ventana de configuración y presionamos **Back to canvas**.
    - Para le memoria presionamos el **+** debajo de "Memory" y seleccionamos la primera opción que dice "Simple Memory". Ignoramos la ventana de configuración y presionamos **Back to canvas**. El resultado final se debería ver como:
    <center>
    <img src='./img/AI_agent2.png' width=20%>
    </center>

    - Hasta ahora lo que está pasando es que nuestro *workflow* va a activarse cuando se reciba un mensaje de WhatsApp. Luego este mensaje de WhatsApp se va a enviar a uno de los LLMs de OpenAI y nos va a dar una respuesta. Lo que falta para finalizar el flujo es enviar la respuesta del modelo a WhatsApp para que la reciba el usuario en su chat. 
    
        Para hacer esto presionamos el **+ a la derecha** de AI Agent, buscamos "WhatsApp" en la barra de busqueda, presionamos "WhatsApp Business Cloud" y seleccionamos la opción "Send Message". Ignoramos la ventana de configuración y presionamos **Back to canvas**. El resultado final se debería ver como:

        <center>
        <img src='./img/complete_workflow.png' width=20%>
        </center>
        
**Felicitaciones!!!** Ahora tienes la estructura completa de un Chatbot equipado con inteligencia artificial.

## Paso 3. Conectar nuestro workflow con las APIs.
Ahora viene el paso más importante, conectar nuestro workflow con WhatsApp y OpenAI. Para hacerlo tenemos que **crear una cuenta de desarrollador en OpenAI y Meta.**

### Paso 3.1. Cuenta de desarrollador en OpenAI.
Esta cuenta es completamente independiente a la cuenta que probablemente muchos ya tengan para usar los servicios de OpenAI. Esta cuenta nos permitirá acceder a las APIs de OpenAI y hacer uso no solo de sus modelos de lenguaje, sino también a sus modelos de transcripción de audio, generación de imagenes y más. 
1. Vamos a ingerar a [OpenAI Platform](https://platform.openai.com/docs/overview) y crearemos una cuenta con el botón **Sign up**.
2. Una vez creada la cuenta y estando en el menú principal, vamos a ir a la sección de **Configuración** presionando el símbolo de tuerca ⚙️ en la esquina superior derecha. 
<center>
    <img src='./img/OpenAI1.png' width=50%>
</center>

3. Luego en el menu de la izquierda, dentro de la sección ORGANIZATION, vamos a buscar la opción que dice **Billing**. 
<center>
    <img src='./img/OpenAI2.png' width=50%>
</center>

4. Luego añadimos un método de pago (Tarjeta de Crédito o Tarjeta Débito MasterCard) seleccionando la opción **Payment methods** y añadimos la información de nuestra tarjeta. 
> Para nuestro desarrollo vamos a utilizar el modelo `gpt-4.1-nano` el cuál tiene un costo de $0.10 dolares por millón de *tokens* en el input y $0.40 dolares por millón de *tokens* en el output. Para una información completa del precio para los demás modelos puede visitar la sección de Pricing de [OpenAI](https://platform.openai.com/docs/pricing). 

<center>
    <img src='./img/OpenAI3.png' width=50%>
</center>

#### ¿Qué es un token?
Un **token** es una pequeña unidad de texto que los modelos de lenguaje (como ChatGPT) usan para procesar lo que escribimos.

- A veces un token es una palabra completa.
- Otras veces es solo una parte de una palabra, un signo de puntuación o incluso un espacio.

Los modelos no entienden el lenguaje como los humanos, así que primero dividen el texto en tokens para poder analizarlo y generar una respuesta. Por ejemplo:

| Texto                          | Tokens | Caracteres |
|-------------------------------|--------|--------|
| `Hola, ¿cómo estás?`          | 6      | 18 |
| `Data Science`                | 2      | 12 |
| `¡Eso es increíblemente útil!`| 7      | 28 |
| `Python es genial.`           | 4      | 17 |

Para tener una noción de cuántos caracteres tiene tu input, puedes usar la herramienta interactiva de OpenAI para [contar tokens](https://platform.openai.com/tokenizer). Para que te hagas una idea, todo lo que se ha escrito en este tutorial hasta aquí 👇 son 10,562 caracteres que equivalen a solo 2,515 tokens. **Usar LLMs es MUY BARATO.**

5. Prosiguiendo con el tutorial. Luego de añadir la información de facturación, vamos a la sección de **API keys** y vamos a darle al botón verde de la esquina superior derecha que dice **Create new secret key**. 
<center>
    <img src='./img/OpenAI4.png' width=50%>
</center>

6. Luego guardamos nuestra `secret key` que, como dice su nombre, debe ser secreta. No se debe compartir este texto con nadie. Esto funciona como una contraseña para acceder a nuestra cuenta de facturación.
<center>
    <img src='./img/OpenAI5.png' width=20%>
</center>

7. Luego vamos a ir a `n8n` otra vez y vamos a presionar el circulo con el logo de OpenAI que dice **OpenAI Chat Model**. Esto nos abrirá la ventana para configurar el modelo. Seleccionamos la opción que dice **Select Credential** debajo de **Credential to connect with**.
<center>
    <img src='./img/OpenAI6.png' width=25%>
</center>

Luego donde dice API key vamos a pegar nuestra `secret key` de OpenAI. Seleccionamos el botón **Save**, cerramos la primera ventana y luego cambiamos el modelo a **gpt-4.1-nano**.

### Paso 3.2. Cuenta de desarrollador en Meta
1. Primero vamos a ingresar con nuestro perfil de Facebook al [Administrador comercial de Meta](https://business.facebook.com/settings/) y seleccionamos la opción **Crear negocio** y lo nombramos **Clase de Python**. 
<center>
    <img src='./img/Meta0.png' width=25%>
</center>

2. Una vez tengamos el negocio creado ubicaremos el botón **Apps** en el panel lateral izquierdo, bajo la sección de **Cuentas**. Seguimos el siguiente paso a paso para crear nuestra App.

<center>
    <img src='./img/Meta1.png' width=25%>
    <img src='./img/Meta2.png' width=25%>
</center>

<center>
    <img src='./img/Meta3.png' width=25%>
    <img src='./img/Meta4.png' width=25%>
</center>

<center>
    <img src='./img/Meta5.png' width=25%>
    <img src='./img/Meta6.png' width=25%>
</center>

<center>
    <img src='./img/Meta7.png' width=25%>
</center>

3. Cuando termines la configuración inicial se abrirá el Panel de apps. Buscamos WhatsApp y presionamos el botón que dice **Configurar**.
<center>
    <img src='./img/Meta8.png' width=25%>
</center>

4. Expandimos la sección que dice **Configuración de la App** en el panel izquierdo. Allí nos situamos dentro de la subsección llamada **Básica**. En esta ventana hay dos números que nos interesan: **Identificador de la app** y **Clave secreta de la app**.

<center>
    <img src='./img/Meta9.png' width=25%>
</center>

5. Vamos a ingresar los valores de **Identificador de la app** y **Clave secreta de la app** en n8n para conectar nuestro agente con la API de Meta.
- Primero seleccionamos el primer elemento del Workflow **WhatsApp Trigger**. 
- Expandimos el menu desplegable debajo del texto **Credential to connect with**.
- Seleccionamos la opción **Create New Credential**.
- Tal como indica la imagen pegamos el valor de **Identificador de la app** en el campo que dice **Client ID** y pegamos el valor de **Clave secreta de la app** en el campo que dice **Client Secret**.
- Presionamos guardar.
<center>
    <img src='./img/wapp_cred1.png' width=25%>
    <img src='./img/wapp_cred2.png' width=25%>
</center>

6. Ahora vamos a configurar la API que se encarga de enviar la respuesta del LLM devuelta a WhatsApp. Dentro de n8n presionamos el último elemento del workflow llamado **WhatsApp Business Cloud**. Seguimos el mismo proceso anterior y seleccionamos la opción de **Create New Credentials** dentro del menu bajo el texto **Credential to connect with**.
<center>
    <img src='./img/wapp_cred3.png' width=25%>
</center>

7. Acá nos pedirá la información de **Access Token** y **Business Account ID** que son llaves completamente independientes a las copiadas anteriormente. Para encontrarlas debemos volver al **Panel de apps de Meta**. Allí debemos expandir la sección llamada **WhatsApp** en el panel lateral izquierdo y luego seleccionar **Configuración de la API**.
- Primero debemos Seleccionar los números de teléfono. Meta proveerá un número para hacer pruebas y es el que seleccionaremos en el menu desplegable de la sección **De**.
<center>
    <img src='./img/Meta10.png' width=25%>
</center>

- Segundo debemos seleccionar un número de teléfono para la sección **Para**. Ahí introduciremos nuestra número de celular con el que usamos WhatsApp. Es importante incluir el código de país (e.g. +57).
- Una vez tengamos los números configurados, seleccionamos el botón **Generar token de acceso**.
<center>
    <img src='./img/Meta11.png' width=25%>
</center>

- Debemos copiar el **Token de acceso** resultante y pegarlo en el cuadro que dice **Access Token** en n8n.
- Luego vamos a copiar el número que dice **Identificador de la cuenta de WhatsApp Business** en el cuadro que dice **Business Account ID** de n8n.
- Presionamos guardar y volvemos al menu de **WhatsApp Business Cloud**.

## Paso 4. Configuraciones finales
1. Dentro del último elemento del Workflow, **WhatsApp Business Cloud**, vamos a asegurarnos de incluir nuestro número de celular con código de país (e.g. 57) en el campo de **Recipient's Phone Number**.
2. Verificamos que el número en **Sender Phone Number (or ID)** coincida con el número de prueba que nos asignó Meta previamente. También debemos asegurarnos de que **Recipient's Phone Number** sea el mismo que definimos en el panel de Meta.
3. En el campo de **Text Body** escribimos `{{ $json.output}}`.
<center>
    <img src='./img/wapp_config1.png' width=25%>
</center>

4. Ahora pasamos a configurar la memoria del agente. Presionamos el elemento llamado **Simpler Memory** del workflow. Seleccionamos la opción **Define below** dentro del menu desplegable debajo del texto **Session ID**. 
5. Escribimos `{{ $('WhatsApp Trigger') }}` en el campo debajo de **Key**.
6. Modificamos el **Context Window Length** por un entero que represente el número de mensajes previos en la conversación que deseamos incluir en la memoria del agente para ser añadidos como contexto de la conversación (e.g. 10).
<center>
    <img src='./img/simple_memory.png' width=25%>
</center>

7. Ahora configuramos el **AI Agent**. Seleccionamos el elemento **AI Agent** dentro del workflow. 
8. Cambiamos el valor de **Source for Prompt (User Message)** por **Define below**.
9. Escribimos `{{ $json.messages[0].text.body }}` dentro del cuadro de dialogo debajo de **Prompt (User Message)**.
<center>
    <img src='./img/AI_agent.png' width=25%>
</center>

## Paso 5. Probar
1. Presionamos el botón **Test workflow**.
2. Le escribimos un mensaje al número de prueba por WhatsApp.
3. Esperamos la respuesta :).